In [0]:
bronze = spark.table("workspace.default.bronze_online_retail")
bronze.printSchema()
display(bronze.limit(5))

In [0]:
from pyspark.sql import functions as F
from datetime import datetime,timezone

INVOICE_COL = "Invoice"
STOCK_COL = "StockCode"
DESC_COL = "Description"
QTY_COL = "Quantity"
DATE_COL = "InvoiceDate"
PRICE_COL = "Price"
CUSTOMER_COL = "Customer_ID"

DQ_TABLE = "workspace.default.dq_results"

run_ts = datetime.now(timezone.utc)
total_rows = bronze.count()

null_customer_id = bronze.filter(F.col(CUSTOMER_COL).isNull()).count()
null_description = bronze.filter(F.col(DESC_COL).isNull()).count()
negative_qty = bronze.filter(F.col(QTY_COL) < 0).count()
zero_or_negative_price = bronze.filter(F.col(PRICE_COL) <= 0).count()
cancelled_invoices = bronze.filter(F.col(INVOICE_COL).startswith("C")).count()

dedup_keys = [INVOICE_COL, STOCK_COL, CUSTOMER_COL, DATE_COL, QTY_COL]
dup_counts = bronze.groupBy(dedup_keys).count().filter(F.col("count") > 1)
duplicate_rows = dup_counts.agg(F.sum(F.col("count") - 1)).collect()[0][0] or 0

dq_checks = [
    ("null_customer_id", null_customer_id),
    ("null_description", null_description),
    ("negative_quantity", negative_qty),
    ("zero_or_negative_price", zero_or_negative_price),
    ("cancelled_invoices", cancelled_invoices),
    ("duplicate_rows", duplicate_rows),
]

dq_df = spark.createDataFrame(
    [(name, count, total_rows, round(count / total_rows * 100, 3), run_ts)
     for name, count in dq_checks],
    ["check_name", "failed_count", "total_rows", "failed_pct", "run_timestamp"],
)

display(dq_df)

In [0]:
DQ_TABLE = "workspace.default.dq_results"

dq_df.write.mode("append").saveAsTable(DQ_TABLE)

print(f"Wrote {dq_df.count()} DQ check rows to {DQ_TABLE}")

In [0]:
silver = (
    bronze
    .withColumn("is_cancelled", F.col(INVOICE_COL).startswith("C"))
    .withColumn("is_return", F.col(QTY_COL) < 0)
    .withColumn("is_null_customer", F.col(CUSTOMER_COL).isNull())
    .withColumn("is_invalid_price", F.col(PRICE_COL) <= 0)
)

display(silver.limit(5))

In [0]:
from pyspark.sql.window import Window

dedup_keys = [INVOICE_COL, STOCK_COL, CUSTOMER_COL, DATE_COL, QTY_COL]

w = Window.partitionBy(*dedup_keys).orderBy(F.lit(1))

silver = (
    silver
    .withColumn("_dup_rank", F.row_number().over(w))
    .withColumn("is_duplicate", F.col("_dup_rank") > 1)
    .drop("_dup_rank")
)

display(silver.filter(F.col("is_duplicate") == True).limit(5))

In [0]:
recheck = silver.agg(
    F.sum(F.col("is_cancelled").cast("int")).alias("is_cancelled_count"),
    F.sum(F.col("is_return").cast("int")).alias("is_return_count"),
    F.sum(F.col("is_null_customer").cast("int")).alias("is_null_customer_count"),
    F.sum(F.col("is_invalid_price").cast("int")).alias("is_invalid_price_count"),
    F.sum(F.col("is_duplicate").cast("int")).alias("is_duplicate_count"),
    F.count("*").alias("total_rows"),
)

display(recheck)

In [0]:
SILVER_TABLE = "workspace.default.silver_online_retail"

silver = silver.withColumn("line_total", F.col(QTY_COL) * F.col(PRICE_COL))

silver.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)

silver_rows = spark.table(SILVER_TABLE).count()
print(f"Silver row count: {silver_rows}")
print(f"Matches bronze row count: {silver_rows == total_rows}")